# E2E-DRO retraining on Colab — portfolio-lab iteration 3

**目的 / Purpose**: retrain Costa & Iyengar's headline nets from scratch to verify
their shipped cache (`cache/exp/*.pkl`) is what training actually produces
(portfolio-lab already cross-validated the *evaluation* of that cache at machine
precision; this verifies the *training*).

**为什么在 Colab / Why Colab**: a clean disposable Linux env — we deliberately do
not install `cvxpylayers` into the local anaconda base. Note: this workload is
**CPU-bound** (cvxpylayers/diffcp solves thousands of small conic programs
sequentially); a GPU runtime is NOT needed — a standard CPU runtime is fine.

**预计耗时 / Runtime**: smoke tests ~minutes; `nom_net` roughly 1–3 h;
`dr_net` roughly 2–6 h (Hellinger layer is heavier). Keep the tab alive, or run
one net per session.

**Robustness**: cell 2 defines `cd_repo()`; every later cell calls it first, so a
kernel restart or out-of-order run can't leave the cwd outside the repo (the
relative paths `./cache/…` and `./new_cache/exp/…` depend on it). If you restart
the runtime, just re-run cells 1→2 then continue.

**Skipped**: the 3×6 lr×epoch CV grid — we extracted the winning hyperparameters
from their cached objects locally (nom: lr=0.02/50ep, dr: lr=0.0125/50ep,
theta: lr=0.0125/40ep, base: lr=0.005/30ep) and pass them directly.

**Caveat**: library versions today are newer than the paper's (torch/cvxpy/
cvxpylayers) — bit-exact reproduction is not expected even with their seed
(=1000). What we check is the *qualitative ranking* and Sharpe within a
reasonable band of the cached reference values.

In [ ]:
# 1) Dependencies (CPU torch that Colab ships is fine)
# ecos: newer cvxpy no longer bundles it, but the layers still request it.
%pip -q install cvxpylayers "cvxpy>=1.4" ecos pandas_datareader alpha_vantage statsmodels psutil
import torch, cvxpy, cvxpylayers
print("torch", torch.__version__, "| cvxpy", cvxpy.__version__)

In [ ]:
# 2) Clone the upstream repo (ships the real dataset in cache/*.pkl)
import os, sys
REPO = "/content/E2E-DRO"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Iyengar-Lab/E2E-DRO.git {REPO}

def cd_repo():
    """Idempotent: make REPO the cwd + importable. Call at the top of every
    cell that touches e2edro/, cache/ or new_cache/ — do NOT rely on %cd
    persisting across cells or a kernel restart (that was the earlier bug)."""
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    os.makedirs("new_cache/exp", exist_ok=True)
    assert os.path.isdir("e2edro") and os.path.isdir("cache"), \
        f"not at repo root, cwd={os.getcwd()}"

cd_repo()
print("cwd:", os.getcwd(), "-> OK")
!ls cache/

In [ ]:
# 3) Load the cached real dataset (no AlphaVantage key needed)
cd_repo()
from e2edro import e2edro as e2e
from e2edro import DataLoad as dl
from e2edro import BaseModels as bm

freq, start, end = 'weekly', '2000-01-01', '2021-09-30'
split, n_obs, n_y = [0.6, 0.4], 104, 20
X, Y = dl.AV(start, end, split, freq=freq, n_obs=n_obs, n_y=n_y,
             use_cache=True, save_results=False, AV_key=None)
n_x, n_y = X.data.shape[1], Y.data.shape[1]
# dl.AV returns TrainTest(X[:-1]), TrainTest(Y[1:]) — the raw 1135 weekly rows
# become 1134 aligned (features_t -> returns_{t+1}). Sanity-check, don't hardcode.
assert X.data.shape[0] == Y.data.shape[0] == 1134, \
    f"unexpected aligned length {X.data.shape[0]} (expected 1134) — upstream data changed?"
assert (n_x, n_y) == (8, 20), f"unexpected feature/asset dims {(n_x, n_y)}"
print("features", X.data.shape, "| assets", Y.data.shape, "-> OK")

In [ ]:
# 4) Shared experiment parameters (verbatim from their main.py)
cd_repo()
import pickle, numpy as np

perf_loss, perf_period = 'sharpe_loss', 13
pred_loss_factor, prisk, dr_layer = 0.5, 'p_var', 'hellinger'
set_seed, cache_path = 1000, './new_cache/exp/'

# gross annualized Sharpe of THEIR cached nets (computed in portfolio-lab iter 2)
REFERENCE = {'ew_net': 1.051, 'po_net': 0.899, 'base_net': 0.721,
             'nom_net': 1.178, 'dr_net': 1.314, 'dr_net_learn_theta': 1.414}

def gross_sharpe(net):
    r = net.portfolio.rets['rets'].to_numpy()
    return r.mean() / r.std(ddof=1) * np.sqrt(52)

def save(net, name):
    with open(cache_path + name + '.pkl', 'wb') as f:
        pickle.dump(net, f, pickle.HIGHEST_PROTOCOL)
    print(f"{name}: retrained Sharpe={gross_sharpe(net):.3f} "
          f"(their cache: {REFERENCE.get(name, float('nan')):.3f}) — saved")

In [ ]:
# 5) SMOKE TEST 1 — equal weight (seconds; exercises the roll-test plumbing)
cd_repo()
ew_net = bm.equal_weight(n_x, n_y, n_obs)
ew_net.net_roll_test(X, Y, n_roll=4)
save(ew_net, 'ew_net')
# expected: matches REFERENCE almost exactly (no training randomness)

In [ ]:
# 6) SMOKE TEST 2 — predict-then-optimize (minutes; exercises the
# CvxpyLayer forward pass, no gradient training)
cd_repo()
po_net = bm.pred_then_opt(n_x, n_y, n_obs, set_seed=set_seed, prisk=prisk).double()
po_net.net_roll_test(X, Y)
save(po_net, 'po_net')

In [ ]:
# 6.5) DIAGNOSTIC — isolate any training failure in <1 min, and project runtime.
# Run this BEFORE committing hours to cells 7/8. The smoke tests above exercise
# the CvxpyLayer FORWARD path only; training adds two untested things: reloading
# the init state, and backprop through the optimization layer.
cd_repo()
import time, traceback, torch
from torch.utils.data import DataLoader
from e2edro import PortfolioClasses as pc

diag = e2e.e2e_net(n_x, n_y, n_obs, prisk=prisk,
                   train_pred=True, train_gamma=True, train_delta=False,
                   set_seed=set_seed, opt_layer='nominal', perf_loss=perf_loss,
                   cache_path=cache_path, perf_period=perf_period,
                   pred_loss_factor=pred_loss_factor).double()
print("A) construct + save init state: OK")

# B) net_roll_test reloads the init state at the top of every roll window.
#    PyTorch >= 2.6 flipped torch.load's weights_only default to True.
try:
    diag.load_state_dict(torch.load(diag.init_state_path))
    print("B) torch.load(init_state): OK (no patch needed)")
except Exception:
    traceback.print_exc()
    diag.load_state_dict(torch.load(diag.init_state_path, weights_only=False))
    print("B) torch.load FAILED with the new default; weights_only=False WORKS "
          "-> run cell 6.6 to patch, then rerun this cell")

# C) one forward + backward through the optimization layer (the untested path)
X.split_update([0.6, 0.1]); Y.split_update([0.6, 0.1])
train_set = DataLoader(pc.SlidingWindow(X.train(), Y.train(), n_obs, perf_period))
n_win = len(train_set)
x, y, y_perf = next(iter(train_set))
try:
    t0 = time.time(); z_star, y_hat = diag(x.squeeze(), y.squeeze()); t_f = time.time() - t0
    loss = diag.perf_loss(z_star, y_perf.squeeze())
    t0 = time.time(); loss.backward(); t_b = time.time() - t0
    print(f"C) forward {t_f:.2f}s + backward {t_b:.2f}s: OK")
    # 50 epochs x windows x 4 roll windows (train set grows ~15% on average)
    steps = 50 * n_win * 4 * 1.15
    print(f"   roll-1 train windows: {n_win} | projected nom_net total: "
          f"{steps * (t_f + t_b) / 3600:.1f} h  <-- CHECK BEFORE PROCEEDING")
except Exception:
    print("C) forward/backward FAILED:")
    traceback.print_exc()

X.split_update(split); Y.split_update(split)   # restore the original split

In [ ]:
# 6.6) PATCH — only run if cell 6.5 reported B) FAILED.
# Restore torch.load's pre-2.6 behaviour for this session. Safe here: the only
# things we load are state dicts this notebook itself just wrote.
import torch
if not getattr(torch, "_e2edro_patched", False):
    _orig_load = torch.load
    def _load(*a, **kw):
        kw.setdefault("weights_only", False)
        return _orig_load(*a, **kw)
    torch.load = _load
    torch._e2edro_patched = True
    print("torch.load patched -> weights_only=False by default")
else:
    print("already patched")

In [ ]:
# 7) Nominal E2E — the first real DFL retraining (~1-3 h CPU)
# CV grid skipped: lr=0.02, epochs=50 were the winners in their cached cv_results.
cd_repo()
nom_net = e2e.e2e_net(n_x, n_y, n_obs, prisk=prisk,
                      train_pred=True, train_gamma=True, train_delta=False,
                      set_seed=set_seed, opt_layer='nominal', perf_loss=perf_loss,
                      cache_path=cache_path, perf_period=perf_period,
                      pred_loss_factor=pred_loss_factor).double()
nom_net.net_roll_test(X, Y, n_roll=4, lr=0.02, epochs=50)
save(nom_net, 'nom_net')

In [ ]:
# 8) DR E2E (Hellinger) — the headline model (~2-6 h CPU)
cd_repo()
dr_net = e2e.e2e_net(n_x, n_y, n_obs, prisk=prisk,
                     train_pred=True, train_gamma=True, train_delta=True,
                     set_seed=set_seed, opt_layer=dr_layer, perf_loss=perf_loss,
                     cache_path=cache_path, perf_period=perf_period,
                     pred_loss_factor=pred_loss_factor).double()
dr_net.net_roll_test(X, Y, n_roll=4, lr=0.0125, epochs=50)
save(dr_net, 'dr_net')

In [ ]:
# 9) (OPTIONAL — uncomment to also retrain the cost-stress winner)
cd_repo()
# dr_net_learn_theta (their main.py Exp 4): DR layer, train_pred=True,
# train_gamma=False, train_delta=False. Winning hyperparams: lr=0.0125, epochs=40.
# dr_net_theta = e2e.e2e_net(n_x, n_y, n_obs, prisk=prisk,
#                            train_pred=True, train_gamma=False, train_delta=False,
#                            set_seed=set_seed, opt_layer=dr_layer, perf_loss=perf_loss,
#                            cache_path=cache_path, perf_period=perf_period,
#                            pred_loss_factor=pred_loss_factor).double()
# dr_net_theta.net_roll_test(X, Y, n_roll=4, lr=0.0125, epochs=40)
# save(dr_net_theta, 'dr_net_learn_theta')

In [ ]:
# 10) Package the retrained pickles for download
cd_repo()
!zip -r retrained_nets.zip new_cache/exp/
from google.colab import files
files.download('retrained_nets.zip')

## 带回本地 / Bring the results home

Unzip into the repo as
`research-projects/portfolio-lab/vendor/E2E-DRO/new_cache/exp/*.pkl`
(do **not** overwrite `cache/exp/` — that is their original artifact), then run:

```bash
python scripts/compare_e2edro_cache.py   # will be extended to diff cache vs new_cache
```

Success criterion: retrained gross Sharpes land near the reference values and
preserve the ranking `dr_net > nom_net > 1/N > po_net`. Large deviations are a
*finding*, not a failure — record either way in `notes.md`.